In [1]:
from ultralytics import YOLO
from pathlib import Path

model_path = Path("../runs/yolo26m.pt")

if not model_path.exists():
    raise FileNotFoundError(f"Modelo não encontrado em {model_path}. Rode o 01_setup.ipynb primeiro.")

model = YOLO(model_path)
print(f"Modelo carregado de: {model_path}")

Modelo carregado de: ..\runs\yolo26m.pt


In [ ]:
import yaml
import tempfile

DATASET_YAML = Path("../../../../data/dataset.yaml")
EPOCHS       = 15
IMGSZ        = 640
BATCH        = 4   # conservador para 4GB VRAM
PROJECT      = "../runs"
RUN_NAME     = "yolo26m-25epochs"

# O Ultralytics exige que "path" no dataset.yaml seja absoluto (senão resolve
# contra a pasta global de datasets do usuário, não a deste repo). Para manter
# o dataset.yaml versionado livre de caminhos específicos de máquina, sobrescrevemos
# "path" aqui em tempo de execução e treinamos a partir de uma cópia resolvida.
with open(DATASET_YAML) as f:
    dataset_cfg = yaml.safe_load(f)

dataset_cfg["path"] = str(DATASET_YAML.parent.resolve())

RESOLVED_DATASET_YAML = Path(tempfile.gettempdir()) / "cytoml_dataset_resolved.yaml"
with open(RESOLVED_DATASET_YAML, "w") as f:
    yaml.safe_dump(dataset_cfg, f)

print(f"Dataset:  {RESOLVED_DATASET_YAML} (path={dataset_cfg['path']})")
print(f"Épocas:   {EPOCHS}")
print(f"Imgsz:    {IMGSZ}")
print(f"Batch:    {BATCH}")
print(f"Saída:    {PROJECT}/{RUN_NAME}")

In [ ]:
import torch
torch.cuda.empty_cache()

results = model.train(
    data=RESOLVED_DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    device=0,        # GPU
    exist_ok=False,   # sobrescreve run anterior com mesmo nome se True.
    # workers=0, # para evitar problemas de multiprocessing em alguns ambientes
)

print("Treino concluído!")
print(f"Melhor modelo salvo em: {results.save_dir}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path(PROJECT) / RUN_NAME

# Plota curvas de treino geradas pelo YOLO
for img_name in ["results.png", "confusion_matrix.png"]:
    img_path = results_dir / img_name
    if img_path.exists():
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(mpimg.imread(img_path))
        ax.axis("off")
        ax.set_title(img_name)
        plt.tight_layout()
        plt.show()